# 2. Realized Semivariance (실현 반분산)

아래 순서로 갈게. 왜 필요한가 → 정의 → 숫자 예시 → 이론적 성질 → 실증 결과 → 함정 → 코드.

---

## 1단계: 왜 필요한가 — RV와 RSkew의 한계

지금까지 본 두 지표에는 각각 약점이 있어.

- **RV** $= \sum r_{t,i}^2$: 제곱하는 순간 부호가 사라져. 앞의 예시처럼 막판 급락한 날과 막판 급등한 날의 RV가 **완전히 같아.**
- **RSkew**: 방향은 잡지만 **3제곱**이라 봉 하나에 극도로 민감해. N = 39에서는 일별 값이 매우 불안정해.

그래서 **제곱(2차)은 유지해서 안정성을 확보하되, 부호 정보는 살리자**는 발상이 나와. 방법은 단순해. **RV를 상승 봉에서 나온 부분과 하락 봉에서 나온 부분으로 쪼개는 거야.**

---

## 2단계: "Semi"variance의 원래 의미

개념 자체는 오래됐어. Markowitz(1959)가 이미 제안한 **하방 반분산(downside semivariance)**이 원조야.

$$
\text{SV}^- = E\left[(X-\mu)^2 \cdot \mathbf{1}\{X < \mu\}\right]
$$

$\mathbf{1}\{\cdot\}$는 **지시함수(indicator function)**야. 괄호 안 조건이 참이면 1, 거짓이면 0이지. 즉 평균보다 **낮은** 값만 골라 분산을 계산하는 거야. 투자자가 싫어하는 건 "위로 튀는 변동"이 아니라 "아래로 튀는 변동"이라는 직관을 반영한 거지. 분산의 **절반(semi)** 쪽만 본다고 해서 semivariance라고 불러.

---

## 3단계: Realized Semivariance의 정의

Barndorff-Nielsen, Kinnebrock, Shephard(2010)가 이 아이디어를 장중 데이터에 적용했어.

$$
RS_t^{+} = \sum_{i=1}^{N} r_{t,i}^2 \cdot \mathbf{1}\{r_{t,i} > 0\}
$$

$$
RS_t^{-} = \sum_{i=1}^{N} r_{t,i}^2 \cdot \mathbf{1}\{r_{t,i} < 0\}
$$

- $RS^+$: **상승한 봉들**의 제곱합 → "좋은 변동성(good volatility)"
- $RS^-$: **하락한 봉들**의 제곱합 → "나쁜 변동성(bad volatility)"

기준점이 평균(μ)이 아니라 **0**인 이유는 앞에서 본 것과 같아. 10분봉 평균은 사실상 0이니까 원점 기준으로 충분해.

그리고 둘을 더하면 정확히 RV가 돼.

$$
RV_t = RS_t^{+} + RS_t^{-}
$$

즉 semivariance는 새로운 걸 만드는 게 아니라, **RV를 부호에 따라 두 조각으로 분해(decomposition)**하는 거야. 수익률이 정확히 0인 봉은 양쪽 어디에도 안 들어가지만, 제곱해도 0이라 합계에는 영향이 없어.

---

## 4단계: 숫자 예시 (앞의 예시 재활용)

**종목 A (조금씩 오르다 막판 급락):** $r = [0.2,\ 0.1,\ 0.1,\ 0.2,\ -1.0]$

- $RS^+ = 0.04 + 0.01 + 0.01 + 0.04 = 0.10$
- $RS^- = 1.00$
- $RV = 1.10$

**종목 B (조금씩 내리다 막판 급등):** $r = [-0.2,\ -0.1,\ -0.1,\ -0.2,\ +1.0]$

- $RS^+ = 1.00$
- $RS^- = 0.10$
- $RV = 1.10$

**종목 C (고르게 오르내림):** $r = [0.5,\ -0.5,\ 0.5,\ -0.5,\ 0.6]$

- $RS^+ = 0.25 + 0.25 + 0.36 = 0.86$
- $RS^- = 0.25 + 0.25 = 0.50$
- $RV = 1.36$

| 종목 | RV | RS⁺ | RS⁻ | RS⁻/RV | RSkew |
|---|---|---|---|---|---|
| A | 1.10 | 0.10 | 1.00 | **91%** | −1.90 |
| B | 1.10 | 1.00 | 0.10 | **9%** | +1.90 |
| C | 1.36 | 0.86 | 0.50 | 37% | 작음 |

A와 B는 RV로는 구분이 안 되지만, semivariance로는 **변동의 91%가 하락에서 왔다**와 **9%만 하락에서 왔다**로 명확히 갈려. RSkew와 같은 정보를 담으면서, 2차라서 훨씬 덜 튀어.

**상대 반분산(relative semivariance)**도 자주 써.

$$
\frac{RS_t^-}{RV_t} \in [0, 1]
$$

종목 간 변동성 크기 차이를 제거한 지표야. 앞에서 배운 **표준화**와 같은 발상이지. 횡단면 비교(종목끼리 비교)를 할 때는 원값보다 이쪽이 적절해.

---

## 5단계: 이론적 성질 — 이게 진짜 핵심

여기서부터가 semivariance가 단순한 "쪼개기" 이상인 이유야.

주가 움직임을 두 종류로 나눠보자.

- **연속 부분(diffusion)**: 매 순간 작게 오르내리는 평상시 움직임
- **점프 부분(jump)**: 실적 발표, 뉴스 같은 이벤트로 갑자기 크게 튀는 움직임

N → ∞일 때 BNKS(2010)가 보인 결과는 다음과 같아.

$$
RS_t^{+} \;\to\; \frac{1}{2} IV_t + \sum_{\text{양의 점프}} J^2
$$

$$
RS_t^{-} \;\to\; \frac{1}{2} IV_t + \sum_{\text{음의 점프}} J^2
$$

$IV_t$(integrated variance)는 연속 부분의 진짜 분산이야.

**해석:** 평상시의 연속적인 움직임은 **상승과 하락에 정확히 반반씩** 나뉘어. 짧은 시간 단위로 보면 오를 확률과 내릴 확률이 대칭이니까. 그래서 $RS^+$와 $RS^-$의 **차이를 만드는 건 오직 점프**야.

이걸 빼면 연속 부분이 깨끗하게 상쇄돼.

$$
RS_t^{+} - RS_t^{-} \;\to\; \sum_{\text{양의 점프}} J^2 - \sum_{\text{음의 점프}} J^2
$$

이게 바로 다음에 배울 **3번 signed jump variation**이야. semivariance를 이해하면 3번은 이 한 줄로 거의 끝나.

---

## 6단계: 실증 결과 — "나쁜 변동성"이 더 중요하다

**(1) 미래 변동성 예측: Patton & Sheppard (2015), "Good Volatility, Bad Volatility"**

HAR 모형(7번 HARQ에서 자세히 다룸)은 기본적으로 이렇게 생겼어.

$$
RV_{t+1} = \beta_0 + \beta_d RV_t + \beta_w RV_t^{(week)} + \beta_m RV_t^{(month)} + \varepsilon_{t+1}
$$

어제, 지난주 평균, 지난달 평균 RV로 내일 RV를 예측하는 구조야. 여기서 어제 RV를 semivariance로 쪼개면:

$$
RV_{t+1} = \beta_0 + \beta^{+} RS_t^{+} + \beta^{-} RS_t^{-} + \beta_w RV_t^{(week)} + \beta_m RV_t^{(month)} + \varepsilon_{t+1}
$$

결과는 $\beta^-$가 $\beta^+$보다 **훨씬 크고**, $RS^+$의 계수는 작거나 유의하지 않았어. 즉 **미래 변동성을 키우는 건 하락에서 온 변동성**이야. 같은 크기의 변동이라도 급락은 이후 불안을 오래 끌고 가고, 급등은 금방 잊혀. 1장에서 말한 레버리지 효과를 장중 단위로 확인한 셈이야.

**(2) 미래 수익률 예측: Bollerslev, Li, Zhao (2020)**

semivariance의 차이(signed jump variation)를 RV로 표준화한 지표가 **다음 주 횡단면 수익률을 예측**한다는 결과야. 이건 3번에서 자세히 다룰게.

---

## 7단계: 네 프로젝트에 쓸 때의 함정

**N = 39의 한계.** 5단계의 이론은 N → ∞일 때 성립해. N = 39면 연속 부분이 정확히 반반으로 나뉘지 않아서, 점프가 없어도 $RS^+ \neq RS^-$가 흔해. 그래서 일별 값보다는 **주 단위 합산**이 더 신뢰할 만해.

**0 수익률 봉.** 저가주나 비유동 종목은 가격이 안 움직인 10분봉이 많아. 이 봉들은 양쪽 어디에도 안 들어가서 정보가 사라져. 0 수익률 비중이 높은 종목은 걸러야 해.

**미시구조 노이즈(bid-ask bounce).** 체결가가 매수호가와 매도호가 사이를 왔다 갔다 하면서 가짜 +/− 수익률이 교대로 생겨. RV를 부풀리고 $RS^+$, $RS^-$를 둘 다 인위적으로 키워. 10분봉이면 1분봉보다는 덜하지만 소형주에서는 여전히 문제야.

**원값 vs 상대값.** 종목 간 비교를 할 때 $RS^-$ 원값을 쓰면 "변동성이 큰 종목"이 뽑힐 뿐이야. 반드시 $RS^-/RV$ 같은 상대값을 쓰거나 RV를 통제해야 해. 앞에서 배운 표준화의 필요성이 여기서 그대로 적용돼.

**첫 봉과 가격제한폭.** RSkew 때와 같은 문제야. 시가 동시호가 봉은 제외하고, 상·하한가 도달일은 표시해둬.

---

## 계산 코드

```python
import numpy as np
import pandas as pd

def realized_semivariance(r: pd.Series) -> pd.Series:
    """r: 하루치 10분봉 로그수익률 (첫 봉/오버나이트 제외)"""
    r = r.dropna()
    rs_pos = (r[r > 0] ** 2).sum()
    rs_neg = (r[r < 0] ** 2).sum()
    rv = rs_pos + rs_neg
    return pd.Series({
        'RV': rv,
        'RS_pos': rs_pos,
        'RS_neg': rs_neg,
        'RS_neg_ratio': rs_neg / rv if rv > 0 else np.nan,
        'zero_ratio': (r == 0).mean(),   # 유동성 필터용
    })

daily = (df.groupby(['ticker', 'date'])['ret10m']
           .apply(realized_semivariance)
           .unstack())

# 주 단위 집계 (노이즈 완화)
daily['week'] = pd.to_datetime(daily.index.get_level_values('date')).to_period('W')
weekly = daily.groupby(['ticker', 'week'])[['RV', 'RS_pos', 'RS_neg']].sum()
weekly['RS_neg_ratio'] = weekly['RS_neg'] / weekly['RV']
```

---

## 네 프로젝트와의 연결

네 원래 질문은 "10분봉 변동성이 다음 날 등락에 시그널을 주는가"였지. semivariance를 쓰면 이 질문이 훨씬 날카로워져.

- 원래 버전: "RV가 크면 내일 오르나 내리나?" → 부호 정보가 없어서 약한 결과가 나올 가능성이 높아.
- 개선 버전: "**오늘 변동성 중 하락 봉의 비중($RS^-/RV$)**이 높으면 내일 또는 다음 주 수익률이 어떻게 되나? 그리고 **그 관계가 고변동 레짐과 저변동 레짐에서 다른가?**"

특히 Patton-Sheppard의 $\beta^- > \beta^+$ 결과가 **레짐별로 다르게 나타나는지**는 그 자체로 좋은 첫 번째 실증 과제야. 변동성 예측은 수익률 예측보다 신호가 훨씬 강해서 결과가 나올 가능성도 높아.

---

**한 줄 요약:** Realized semivariance는 RV를 상승 봉의 제곱합($RS^+$)과 하락 봉의 제곱합($RS^-$)으로 쪼갠 거야. 평상시 움직임은 양쪽에 반반 나뉘므로, 둘의 차이는 **점프의 방향**을 드러내. 실증적으로는 하락 쪽 변동성($RS^-$)이 미래 변동성을 훨씬 강하게 예측해.

다음 3번 **signed jump variation**은 5단계의 $RS^+ - RS^-$를 본격적으로 다뤄. 준비되면 말해줘.